🔄 **DeepL Translation**

The dataset is translated to English using the DeepL API to ensure compatibility with English-based NLP models and to standardize the text input.

In [4]:
from langdetect import detect, DetectorFactory
from tqdm import tqdm
import pandas as pd
import deepl
import re

DEEPL_API_KEY = "fac2579b-4263-4c36-b124-e57c87cfc512:fx"

tqdm.pandas()

In [ ]:
DetectorFactory.seed=0

translator=deepl.Translator(DEEPL_API_KEY)

In [6]:
df = pd.read_csv('./Data/cyberbullying_tweets.csv')
df=df.dropna(how='any')
df.head()

,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


In [7]:
supported_languages = [
    'AR', 'BG', 'CS', 'DA', 'DE', 'EL', 'EN', 'ES', 'ET', 'FI', 'FR',
    'HU', 'ID', 'IT', 'JA', 'KO', 'LT', 'LV', 'NB', 'NL', 'PL', 'PT',
    'RO', 'RU', 'SK', 'SL', 'SV', 'TR', 'UK', 'ZH'
]

In [8]:
def detect_translate(text):
    cleaned_text = re.sub(r'http\S+|www\S+|https\S+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '', text)
    cleaned_text = re.sub(r'#\w+', '', cleaned_text)
    cleaned_text = re.sub(r'[^\w\s]', '', cleaned_text)

    if not cleaned_text.strip():
        return text

    try:
        lang = detect(cleaned_text)

        if lang != 'en' and lang.upper() in supported_languages:
            try:
                translated = translator.translate_text(text, source_lang=lang.upper(), target_lang="EN-US")
                return translated.text
            except Exception as e:
                print(f"{cleaned_text}")
                print(f"{text}")
                print(f"Error during translation: {e}")
                return "not_translated"
        else:
            return text
    except Exception as e:
        print(f"{cleaned_text}")
        print(f"{text}")
        print(f"Error during language detection: {e}")
        return text


In [11]:
text="Kids Love😘❤ @ Mohamad Bin Zayed City مدينة محمد بن زايد "
lang=detect_translate(text)
print(f"{lang}")

Kids Love😘❤ @ Mohamad Bin Zayed City مدينة محمد بن زايد 


**Make The Label Dictionary**

In [ ]:
df['tweet_text'] = df['tweet_text'].progress_apply(detect_translate)

In [ ]:
labels = df['cyberbullying_type'].unique()

label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
df['label_ID'] = df['cyberbullying_type'].map(label2id)
df.head()

In [ ]:
df.to_csv('./Data/translated_tweet_comments.csv',index=False)